In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NAMESPACE = f"{CATALOG}.{SCHEMA}"

SENSOR_ENRICHED = f"{NAMESPACE}.silver_sensor_enriched"

print("Using:", SENSOR_ENRICHED)

df = spark.table(SENSOR_ENRICHED)

print("Row count:", df.count())

df.printSchema()

display(
    df.orderBy("shipment_id", "event_time")
)

In [0]:
from pyspark.sql import functions as F

# ============================================================
# GOLD 1: SHIPMENT TEMPERATURE WINDOWS
# ============================================================

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NAMESPACE = f"{CATALOG}.{SCHEMA}"

SOURCE_TABLE = f"{NAMESPACE}.silver_sensor_enriched"
TARGET_TABLE = f"{NAMESPACE}.shipment_temperature_windows"

RUN_ID = "manual_gold_001"

print("================================================")
print("BUILDING GOLD: SHIPMENT TEMPERATURE WINDOWS")
print("================================================")
print("Source :", SOURCE_TABLE)
print("Target :", TARGET_TABLE)

# ------------------------------------------------------------
# 1. Read enriched Silver events
# ------------------------------------------------------------

sensor_df = spark.table(SOURCE_TABLE)

# ------------------------------------------------------------
# 2. Validate required columns
# ------------------------------------------------------------

required_columns = {
    "shipment_id",
    "event_time",
    "temperature_c",
    "is_temperature_exception"
}

missing_columns = required_columns - set(sensor_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

# ------------------------------------------------------------
# 3. Build 5-minute event-time windows
# ------------------------------------------------------------

windowed_df = (
    sensor_df
    .groupBy(
        "shipment_id",
        F.window(
            F.col("event_time"),
            "5 minutes"
        )
    )
    .agg(
        F.min("temperature_c").alias(
            "minimum_temperature_c"
        ),

        F.max("temperature_c").alias(
            "maximum_temperature_c"
        ),

        F.count("*").alias(
            "accepted_reading_count"
        ),

        F.sum(
            F.when(
                F.col("is_temperature_exception") == True,
                F.lit(1)
            ).otherwise(F.lit(0))
        ).alias(
            "exception_count"
        )
    )
)

# ------------------------------------------------------------
# 4. Flatten Spark window structure
# ------------------------------------------------------------

gold_df = (
    windowed_df

    .withColumn(
        "window_start",
        F.col("window.start")
    )

    .withColumn(
        "window_end",
        F.col("window.end")
    )

    .drop("window")
)

# ------------------------------------------------------------
# 5. Calculate exception rate
# ------------------------------------------------------------

gold_df = (
    gold_df
    .withColumn(
        "exception_rate",
        F.when(
            F.col("accepted_reading_count") > 0,

            (
                F.col("exception_count").cast("double")
                /
                F.col("accepted_reading_count").cast("double")
            ) * F.lit(100.0)
        )
        .otherwise(
            F.lit(None).cast("double")
        )
    )
)

# ------------------------------------------------------------
# 6. Data-status field
# ------------------------------------------------------------

gold_df = (
    gold_df
    .withColumn(
        "data_status",
        F.when(
            F.col("accepted_reading_count") > 0,
            F.lit("HAS_DATA")
        )
        .otherwise(
            F.lit("NO_DATA")
        )
    )
)

# ------------------------------------------------------------
# 7. Operational metadata
# ------------------------------------------------------------

gold_df = (
    gold_df
    .withColumn(
        "evaluated_at",
        F.current_timestamp()
    )
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
)

# ------------------------------------------------------------
# 8. Final column ordering
# ------------------------------------------------------------

gold_df = gold_df.select(
    "shipment_id",
    "window_start",
    "window_end",
    "minimum_temperature_c",
    "maximum_temperature_c",
    "accepted_reading_count",
    "exception_count",
    "exception_rate",
    "data_status",
    "evaluated_at",
    "run_id"
)

# ------------------------------------------------------------
# 9. Write Gold Delta table
# ------------------------------------------------------------

(
    gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print()
print("GOLD TABLE CREATED SUCCESSFULLY")
print("Table:", TARGET_TABLE)
print("Rows :", spark.table(TARGET_TABLE).count())

In [0]:
display(
    spark.table(
        "training_lab_classic.cold_chain.shipment_temperature_windows"
    )
    .orderBy(
        "shipment_id",
        "window_start"
    )
)

In [0]:
from pyspark.sql import functions as F

GOLD_TABLE = (
    "training_lab_classic."
    "cold_chain."
    "shipment_temperature_windows"
)

gold = spark.table(GOLD_TABLE)

print(
    "================================================"
)
print(
    "GOLD TEMPERATURE WINDOW VALIDATION"
)
print(
    "================================================"
)

# ------------------------------------------------------------
# Basic statistics
# ------------------------------------------------------------

window_count = gold.count()

shipment_count = (
    gold
    .select("shipment_id")
    .distinct()
    .count()
)

total_readings = (
    gold
    .agg(
        F.sum("accepted_reading_count")
        .alias("total")
    )
    .first()["total"]
)

total_exceptions = (
    gold
    .agg(
        F.sum("exception_count")
        .alias("total")
    )
    .first()["total"]
)

print("Gold windows             :", window_count)
print("Shipments represented    :", shipment_count)
print("Accepted readings        :", total_readings)
print("Temperature exceptions   :", total_exceptions)

# ------------------------------------------------------------
# Validation 1:
# All accepted Silver records must be represented once
# across the Gold windows.
# ------------------------------------------------------------

silver_count = (
    spark.table(
        "training_lab_classic."
        "cold_chain."
        "silver_sensor_enriched"
    )
    .count()
)

assert total_readings == silver_count, (
    f"Reading reconciliation failed: "
    f"Gold={total_readings}, Silver={silver_count}"
)

# ------------------------------------------------------------
# Validation 2:
# Fixture should contain the previously proven exceptions.
# ------------------------------------------------------------

assert total_exceptions == 3, (
    f"Expected 3 exceptions, found {total_exceptions}"
)

# ------------------------------------------------------------
# Validation 3:
# Exception rate cannot be negative or above 100.
# ------------------------------------------------------------

invalid_rates = (
    gold
    .filter(
        (F.col("exception_rate") < 0)
        |
        (F.col("exception_rate") > 100)
    )
    .count()
)

assert invalid_rates == 0, (
    f"Found {invalid_rates} invalid exception rates"
)

# ------------------------------------------------------------
# Validation 4:
# Window boundaries must make sense.
# ------------------------------------------------------------

invalid_windows = (
    gold
    .filter(
        F.col("window_end") <= F.col("window_start")
    )
    .count()
)

assert invalid_windows == 0, (
    f"Found {invalid_windows} invalid windows"
)

# ------------------------------------------------------------
# Validation 5:
# Counts cannot be negative.
# ------------------------------------------------------------

invalid_counts = (
    gold
    .filter(
        (F.col("accepted_reading_count") < 0)
        |
        (F.col("exception_count") < 0)
        |
        (
            F.col("exception_count")
            >
            F.col("accepted_reading_count")
        )
    )
    .count()
)

assert invalid_counts == 0, (
    f"Found {invalid_counts} invalid count records"
)

print()
print("===============================================")
print("GOLD TEMPERATURE WINDOWS: PASS")
print("===============================================")

display(
    gold.orderBy(
        "shipment_id",
        "window_start"
    )
)

In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NAMESPACE = f"{CATALOG}.{SCHEMA}"

required_tables = [
    "silver_shipments",
    "silver_sensor_enriched",
    "silver_delivery_events"
]

print("==============================================")
print("PHASE 8B SOURCE TABLE CHECK")
print("==============================================")

for table_name in required_tables:
    full_name = f"{NAMESPACE}.{table_name}"

    if not spark.catalog.tableExists(full_name):
        raise ValueError(f"Required table does not exist: {full_name}")

    row_count = spark.table(full_name).count()

    print(f"{table_name}: {row_count} rows")

print()
print("All Phase 8B source tables are available.")

In [0]:
from pyspark.sql import functions as F

shipments = spark.table(
    "training_lab_classic.cold_chain.silver_shipments"
)

sensors = spark.table(
    "training_lab_classic.cold_chain.silver_sensor_enriched"
)

deliveries = spark.table(
    "training_lab_classic.cold_chain.silver_delivery_events"
)

print("==============================================")
print("TIMESTAMP RANGE INSPECTION")
print("==============================================")

print("Shipment time range")

display(
    shipments.agg(
        F.min("dispatch_time").alias("earliest_dispatch"),
        F.max("dispatch_time").alias("latest_dispatch"),
        F.min("expected_arrival").alias("earliest_expected_arrival"),
        F.max("expected_arrival").alias("latest_expected_arrival")
    )
)

print("Sensor time range")

display(
    sensors.agg(
        F.min("event_time").alias("earliest_sensor_event"),
        F.max("event_time").alias("latest_sensor_event")
    )
)

print("Delivery time range")

display(
    deliveries.agg(
        F.min("event_time").alias("earliest_delivery_event"),
        F.max("event_time").alias("latest_delivery_event")
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NAMESPACE = f"{CATALOG}.{SCHEMA}"

SHIPMENTS_TABLE = f"{NAMESPACE}.silver_shipments"
SENSOR_TABLE = f"{NAMESPACE}.silver_sensor_enriched"
DELIVERY_TABLE = f"{NAMESPACE}.silver_delivery_events"
TARGET_TABLE = f"{NAMESPACE}.shipment_health"

RUN_ID = "manual_gold_001"
STALE_THRESHOLD_MINUTES = 10

spark.conf.set("spark.sql.session.timeZone", "UTC")

print("==============================================")
print("BUILDING GOLD: SHIPMENT HEALTH")
print("==============================================")
print("Shipments :", SHIPMENTS_TABLE)
print("Sensors   :", SENSOR_TABLE)
print("Deliveries:", DELIVERY_TABLE)
print("Target    :", TARGET_TABLE)

shipments_df = spark.table(SHIPMENTS_TABLE)

sensor_df = spark.table(SENSOR_TABLE)

delivery_df = spark.table(DELIVERY_TABLE)

required_shipment_columns = {
    "shipment_id",
    "product_id",
    "vehicle_id",
    "origin",
    "destination",
    "dispatch_time",
    "expected_arrival"
}

required_sensor_columns = {
    "event_id",
    "shipment_id",
    "event_time",
    "temperature_c",
    "is_temperature_exception"
}

required_delivery_columns = {
    "shipment_id",
    "status",
    "event_time"
}

missing_shipment_columns = (
    required_shipment_columns - set(shipments_df.columns)
)

missing_sensor_columns = (
    required_sensor_columns - set(sensor_df.columns)
)

missing_delivery_columns = (
    required_delivery_columns - set(delivery_df.columns)
)

if missing_shipment_columns:
    raise ValueError(
        f"Missing shipment columns: "
        f"{sorted(missing_shipment_columns)}"
    )

if missing_sensor_columns:
    raise ValueError(
        f"Missing sensor columns: "
        f"{sorted(missing_sensor_columns)}"
    )

if missing_delivery_columns:
    raise ValueError(
        f"Missing delivery columns: "
        f"{sorted(missing_delivery_columns)}"
    )

as_of_time = (
    shipments_df
    .agg(
        (
            F.max("expected_arrival")
            + F.expr("INTERVAL 1 HOUR")
        ).alias("as_of_time")
    )
    .first()["as_of_time"]
)

if as_of_time is None:
    raise ValueError(
        "Could not derive as_of_time from expected_arrival."
    )

print("Evaluation time:", as_of_time)
print(
    "Stale threshold:",
    STALE_THRESHOLD_MINUTES,
    "minutes"
)

latest_sensor_window = (
    Window
    .partitionBy("shipment_id")
    .orderBy(
        F.col("event_time").desc(),
        F.col("event_id").desc()
    )
)

latest_sensor_df = (
    sensor_df
    .withColumn(
        "_latest_sensor_rank",
        F.row_number().over(latest_sensor_window)
    )
    .filter(
        F.col("_latest_sensor_rank") == 1
    )
    .select(
        "shipment_id",
        F.col("temperature_c").alias(
            "latest_temperature_c"
        ),
        F.col("event_time").alias(
            "latest_event_time"
        ),
        F.col("is_temperature_exception").alias(
            "latest_is_temperature_exception"
        )
    )
)

delivered_shipments_df = (
    delivery_df
    .filter(
        F.upper(F.col("status")) == "DELIVERED"
    )
    .groupBy("shipment_id")
    .agg(
        F.max("event_time").alias(
            "actual_arrival"
        )
    )
)

shipment_health_df = (
    shipments_df.alias("s")
    .join(
        latest_sensor_df.alias("ls"),
        on="shipment_id",
        how="left"
    )
    .join(
        delivered_shipments_df.alias("d"),
        on="shipment_id",
        how="left"
    )
)

shipment_health_df = (
    shipment_health_df
    .withColumn(
        "evaluated_at",
        F.lit(as_of_time).cast("timestamp")
    )
    .withColumn(
        "active_shipment_status",
        F.when(
            F.col("actual_arrival").isNull(),
            F.lit("ACTIVE")
        ).otherwise(
            F.lit("DELIVERED")
        )
    )
    .withColumn(
        "current_exception_status",
        F.when(
            F.col("latest_event_time").isNull(),
            F.lit("NO_ACCEPTED_READING")
        )
        .when(
            F.col("latest_is_temperature_exception") == True,
            F.lit("TEMPERATURE_EXCEPTION")
        )
        .otherwise(
            F.lit("NORMAL")
        )
    )
    .withColumn(
        "sensor_age_minutes",
        F.when(
            F.col("latest_event_time").isNotNull(),
            (
                F.unix_timestamp("evaluated_at")
                - F.unix_timestamp("latest_event_time")
            ) / F.lit(60.0)
        ).otherwise(
            F.lit(None).cast("double")
        )
    )
    .withColumn(
        "missing_sensor_status",
        F.when(
            F.col("active_shipment_status") != "ACTIVE",
            F.lit("NOT_APPLICABLE")
        )
        .when(
            F.col("latest_event_time").isNull(),
            F.lit("MISSING")
        )
        .when(
            F.col("latest_event_time")
            < (
                F.col("evaluated_at")
                - F.expr(
                    f"INTERVAL {STALE_THRESHOLD_MINUTES} MINUTES"
                )
            ),
            F.lit("STALE")
        )
        .otherwise(
            F.lit("RECENT")
        )
    )
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
)

shipment_health_df = shipment_health_df.select(
    "shipment_id",
    "product_id",
    "vehicle_id",
    "origin",
    "destination",
    "dispatch_time",
    "expected_arrival",
    "actual_arrival",
    "latest_temperature_c",
    "latest_event_time",
    "sensor_age_minutes",
    "current_exception_status",
    "active_shipment_status",
    "missing_sensor_status",
    "evaluated_at",
    "run_id"
)

(
    shipment_health_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print()
print("SHIPMENT HEALTH CREATED SUCCESSFULLY")
print("Table:", TARGET_TABLE)
print("Rows :", spark.table(TARGET_TABLE).count())

display(
    spark.table(TARGET_TABLE)
    .orderBy("shipment_id")
)

In [0]:
from pyspark.sql import functions as F

NAMESPACE = "training_lab_classic.cold_chain"

SHIPMENTS_TABLE = f"{NAMESPACE}.silver_shipments"
HEALTH_TABLE = f"{NAMESPACE}.shipment_health"

shipments = spark.table(SHIPMENTS_TABLE)
health = spark.table(HEALTH_TABLE)

print("==============================================")
print("SHIPMENT HEALTH VALIDATION")
print("==============================================")

shipment_count = shipments.count()
health_count = health.count()

print("Silver shipments :", shipment_count)
print("Health rows      :", health_count)

assert health_count == shipment_count, (
    f"Expected {shipment_count} health rows, "
    f"found {health_count}"
)

duplicate_health_shipments = (
    health
    .groupBy("shipment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_health_shipments == 0, (
    "Duplicate shipment IDs found in shipment_health"
)

invalid_active_status = (
    health
    .filter(
        ~F.col("active_shipment_status").isin(
            "ACTIVE",
            "DELIVERED"
        )
    )
    .count()
)

assert invalid_active_status == 0, (
    "Invalid active_shipment_status found"
)

invalid_sensor_status = (
    health
    .filter(
        ~F.col("missing_sensor_status").isin(
            "MISSING",
            "STALE",
            "RECENT",
            "NOT_APPLICABLE"
        )
    )
    .count()
)

assert invalid_sensor_status == 0, (
    "Invalid missing_sensor_status found"
)

delivered_with_active_status = (
    health
    .filter(
        F.col("actual_arrival").isNotNull()
        & (F.col("active_shipment_status") == "ACTIVE")
    )
    .count()
)

assert delivered_with_active_status == 0, (
    "Delivered shipments incorrectly classified as ACTIVE"
)

active_with_not_applicable = (
    health
    .filter(
        (F.col("active_shipment_status") == "ACTIVE")
        & (F.col("missing_sensor_status") == "NOT_APPLICABLE")
    )
    .count()
)

assert active_with_not_applicable == 0, (
    "Active shipment has NOT_APPLICABLE sensor status"
)

sh003 = (
    health
    .filter(F.col("shipment_id") == "SH003")
    .collect()
)

assert len(sh003) == 1, (
    "SH003 was not found exactly once"
)

assert sh003[0]["active_shipment_status"] == "ACTIVE", (
    "SH003 should remain ACTIVE because it has no delivery"
)

assert sh003[0]["actual_arrival"] is None, (
    "SH003 must not have a manufactured actual arrival"
)

print()
print("Status summary")

display(
    health
    .groupBy(
        "active_shipment_status",
        "missing_sensor_status"
    )
    .count()
    .orderBy(
        "active_shipment_status",
        "missing_sensor_status"
    )
)

print()
print("==============================================")
print("SHIPMENT HEALTH: PASS")
print("==============================================")

display(
    health.orderBy("shipment_id")
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NAMESPACE = f"{CATALOG}.{SCHEMA}"

SHIPMENTS_TABLE = f"{NAMESPACE}.silver_shipments"
DELIVERY_TABLE = f"{NAMESPACE}.silver_delivery_events"
TARGET_TABLE = f"{NAMESPACE}.delivery_performance"

RUN_ID = "manual_gold_001"

spark.conf.set("spark.sql.session.timeZone", "UTC")

print("==============================================")
print("BUILDING GOLD: DELIVERY PERFORMANCE")
print("==============================================")
print("Shipments :", SHIPMENTS_TABLE)
print("Deliveries:", DELIVERY_TABLE)
print("Target    :", TARGET_TABLE)

shipments_df = spark.table(SHIPMENTS_TABLE)

delivery_df = spark.table(DELIVERY_TABLE)

required_shipment_columns = {
    "shipment_id",
    "origin",
    "destination",
    "expected_arrival"
}

required_delivery_columns = {
    "event_id",
    "shipment_id",
    "status",
    "event_time"
}

missing_shipment_columns = (
    required_shipment_columns - set(shipments_df.columns)
)

missing_delivery_columns = (
    required_delivery_columns - set(delivery_df.columns)
)

if missing_shipment_columns:
    raise ValueError(
        f"Missing shipment columns: "
        f"{sorted(missing_shipment_columns)}"
    )

if missing_delivery_columns:
    raise ValueError(
        f"Missing delivery columns: "
        f"{sorted(missing_delivery_columns)}"
    )

as_of_time = (
    shipments_df
    .agg(
        (
            F.max("expected_arrival")
            + F.expr("INTERVAL 1 HOUR")
        ).alias("as_of_time")
    )
    .first()["as_of_time"]
)

if as_of_time is None:
    raise ValueError(
        "Could not derive as_of_time from expected_arrival."
    )

print("Evaluation time:", as_of_time)

delivered_events_df = (
    delivery_df
    .filter(
        F.upper(F.col("status")) == "DELIVERED"
    )
)

latest_delivery_window = (
    Window
    .partitionBy("shipment_id")
    .orderBy(
        F.col("event_time").desc(),
        F.col("event_id").desc()
    )
)

latest_delivery_df = (
    delivered_events_df
    .withColumn(
        "_delivery_rank",
        F.row_number().over(latest_delivery_window)
    )
    .filter(
        F.col("_delivery_rank") == 1
    )
    .select(
        "shipment_id",
        F.col("event_time").alias(
            "actual_arrival"
        )
    )
)

delivery_performance_df = (
    shipments_df.alias("s")
    .join(
        latest_delivery_df.alias("d"),
        on="shipment_id",
        how="left"
    )
    .withColumn(
        "evaluated_at",
        F.lit(as_of_time).cast("timestamp")
    )
)

delivery_performance_df = (
    delivery_performance_df
    .withColumn(
        "delivery_status",
        F.when(
            F.col("actual_arrival").isNull(),
            F.lit("UNDELIVERED")
        )
        .when(
            F.col("actual_arrival")
            > F.col("expected_arrival"),
            F.lit("DELIVERED_LATE")
        )
        .otherwise(
            F.lit("DELIVERED_ON_TIME")
        )
    )
    .withColumn(
        "delay_minutes",
        F.when(
            F.col("actual_arrival").isNull(),
            F.lit(None).cast("long")
        )
        .otherwise(
            F.greatest(
                F.lit(0),
                (
                    (
                        F.unix_timestamp("actual_arrival")
                        - F.unix_timestamp("expected_arrival")
                    ) / F.lit(60)
                ).cast("long")
            )
        )
    )
    .withColumn(
        "open_overdue_status",
        F.when(
            F.col("actual_arrival").isNull()
            & (
                F.col("evaluated_at")
                > F.col("expected_arrival")
            ),
            F.lit("OPEN_OVERDUE")
        )
        .when(
            F.col("actual_arrival").isNull(),
            F.lit("OPEN_NOT_OVERDUE")
        )
        .otherwise(
            F.lit("NOT_APPLICABLE")
        )
    )
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
)

delivery_performance_df = delivery_performance_df.select(
    "shipment_id",
    "origin",
    "destination",
    "expected_arrival",
    "actual_arrival",
    "delivery_status",
    "delay_minutes",
    "open_overdue_status",
    "evaluated_at",
    "run_id"
)

(
    delivery_performance_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print()
print("DELIVERY PERFORMANCE CREATED SUCCESSFULLY")
print("Table:", TARGET_TABLE)
print("Rows :", spark.table(TARGET_TABLE).count())

display(
    spark.table(TARGET_TABLE)
    .orderBy("shipment_id")
)

In [0]:
from pyspark.sql import functions as F

DELIVERY_PERFORMANCE_TABLE = (
    "training_lab_classic."
    "cold_chain."
    "delivery_performance"
)

delivery = spark.table(DELIVERY_PERFORMANCE_TABLE)

print("==============================================")
print("DELIVERY PERFORMANCE VALIDATION")
print("==============================================")

print("Total rows:", delivery.count())

assert delivery.count() == 5, (
    f"Expected 5 shipment rows, found {delivery.count()}"
)

duplicate_shipments = (
    delivery
    .groupBy("shipment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_shipments == 0, (
    "Duplicate shipments found in delivery_performance"
)

negative_delays = (
    delivery
    .filter(F.col("delay_minutes") < 0)
    .count()
)

assert negative_delays == 0, (
    f"Found {negative_delays} negative delay records"
)

undelivered_with_arrival = (
    delivery
    .filter(
        (F.col("delivery_status") == "UNDELIVERED")
        & F.col("actual_arrival").isNotNull()
    )
    .count()
)

assert undelivered_with_arrival == 0, (
    "Undelivered shipment has an actual arrival"
)

delivered_without_arrival = (
    delivery
    .filter(
        F.col("delivery_status").startswith("DELIVERED")
        & F.col("actual_arrival").isNull()
    )
    .count()
)

assert delivered_without_arrival == 0, (
    "Delivered shipment is missing actual arrival"
)

expected_delays = {
    "SH001": 30,
    "SH002": 0,
    "SH004": 0,
    "SH005": 60
}

actual_delays = {
    row["shipment_id"]: row["delay_minutes"]
    for row in (
        delivery
        .filter(
            F.col("shipment_id").isin(
                list(expected_delays.keys())
            )
        )
        .select(
            "shipment_id",
            "delay_minutes"
        )
        .collect()
    )
}

for shipment_id, expected_delay in expected_delays.items():
    actual_delay = actual_delays.get(shipment_id)

    assert actual_delay == expected_delay, (
        f"{shipment_id}: expected delay "
        f"{expected_delay}, found {actual_delay}"
    )

sh003_rows = (
    delivery
    .filter(F.col("shipment_id") == "SH003")
    .collect()
)

assert len(sh003_rows) == 1, (
    "SH003 was not found exactly once"
)

sh003 = sh003_rows[0]

assert sh003["actual_arrival"] is None, (
    "SH003 actual arrival must remain NULL"
)

assert sh003["delivery_status"] == "UNDELIVERED", (
    "SH003 should be UNDELIVERED"
)

assert sh003["delay_minutes"] is None, (
    "SH003 delay should be NULL because it was not delivered"
)

assert sh003["open_overdue_status"] == "OPEN_OVERDUE", (
    "SH003 should be OPEN_OVERDUE for the selected evaluation time"
)

print()
print("Verified delivery delays:")
print("SH001 = 30")
print("SH002 = 0")
print("SH004 = 0")
print("SH005 = 60")
print("SH003 actual arrival = NULL")
print("SH003 status = OPEN_OVERDUE")

print()
print("==============================================")
print("DELIVERY PERFORMANCE: PASS")
print("==============================================")

display(
    delivery.orderBy("shipment_id")
)


In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NAMESPACE = f"{CATALOG}.{SCHEMA}"

TABLES = {
    "bronze_sensors": (
        f"{NAMESPACE}.bronze_sensor_events"
    ),
    "silver_sensors": (
        f"{NAMESPACE}.silver_sensor_events"
    ),
    "enriched_sensors": (
        f"{NAMESPACE}.silver_sensor_enriched"
    ),
    "quarantine_sensors": (
        f"{NAMESPACE}.quarantine_sensor_events"
    ),
    "silver_shipments": (
        f"{NAMESPACE}.silver_shipments"
    ),
    "silver_products": (
        f"{NAMESPACE}.silver_product_limits"
    ),
    "silver_deliveries": (
        f"{NAMESPACE}.silver_delivery_events"
    ),
    "temperature_windows": (
        f"{NAMESPACE}.shipment_temperature_windows"
    ),
    "shipment_health": (
        f"{NAMESPACE}.shipment_health"
    ),
    "delivery_performance": (
        f"{NAMESPACE}.delivery_performance"
    )
}

print("==============================================")
print("FULL GOLD ACCEPTANCE TEST")
print("==============================================")

print()
print("1. TABLE EXISTENCE")
print("----------------------------------------------")

for logical_name, table_name in TABLES.items():
    exists = spark.catalog.tableExists(table_name)

    print(f"{logical_name}: {exists}")

    assert exists, (
        f"Required table does not exist: {table_name}"
    )

bronze_sensors = spark.table(
    TABLES["bronze_sensors"]
)

silver_sensors = spark.table(
    TABLES["silver_sensors"]
)

enriched_sensors = spark.table(
    TABLES["enriched_sensors"]
)

quarantine_sensors = spark.table(
    TABLES["quarantine_sensors"]
)

silver_shipments = spark.table(
    TABLES["silver_shipments"]
)

silver_products = spark.table(
    TABLES["silver_products"]
)

silver_deliveries = spark.table(
    TABLES["silver_deliveries"]
)

temperature_windows = spark.table(
    TABLES["temperature_windows"]
)

shipment_health = spark.table(
    TABLES["shipment_health"]
)

delivery_performance = spark.table(
    TABLES["delivery_performance"]
)

print()
print("2. EXPECTED FIXTURE COUNTS")
print("----------------------------------------------")

counts = {
    "bronze_sensors": bronze_sensors.count(),
    "silver_sensors": silver_sensors.count(),
    "enriched_sensors": enriched_sensors.count(),
    "quarantine_sensors": quarantine_sensors.count(),
    "silver_shipments": silver_shipments.count(),
    "silver_products": silver_products.count(),
    "silver_deliveries": silver_deliveries.count(),
    "shipment_health": shipment_health.count(),
    "delivery_performance": delivery_performance.count()
}

for object_name, row_count in counts.items():
    print(f"{object_name}: {row_count}")

assert counts["bronze_sensors"] == 20
assert counts["silver_sensors"] == 16
assert counts["enriched_sensors"] == 16
assert counts["quarantine_sensors"] == 3
assert counts["silver_shipments"] == 5
assert counts["silver_products"] == 3
assert counts["silver_deliveries"] == 4
assert counts["shipment_health"] == 5
assert counts["delivery_performance"] == 5

print()
print("3. SENSOR RECONCILIATION")
print("----------------------------------------------")

e015_bronze_count = (
    bronze_sensors
    .filter(F.col("event_id") == "E015")
    .count()
)

e015_silver_count = (
    silver_sensors
    .filter(F.col("event_id") == "E015")
    .count()
)

duplicate_copies_removed = (
    e015_bronze_count - e015_silver_count
)

reconciled_total = (
    counts["silver_sensors"]
    + counts["quarantine_sensors"]
    + duplicate_copies_removed
)

print("Bronze sensors           :", counts["bronze_sensors"])
print("Silver sensors           :", counts["silver_sensors"])
print("Quarantine sensors       :", counts["quarantine_sensors"])
print("Duplicate copies removed :", duplicate_copies_removed)
print("Reconciled total         :", reconciled_total)

assert e015_bronze_count == 2
assert e015_silver_count == 1
assert duplicate_copies_removed == 1

assert reconciled_total == counts["bronze_sensors"], (
    "Sensor reconciliation failed"
)

print()
print("4. TEMPERATURE EXCEPTION RULES")
print("----------------------------------------------")

expected_exception_status = {
    "E004": True,
    "E008": True,
    "E011": True,
    "E003": False,
    "E007": False
}

actual_exception_rows = (
    enriched_sensors
    .filter(
        F.col("event_id").isin(
            list(expected_exception_status.keys())
        )
    )
    .select(
        "event_id",
        "is_temperature_exception"
    )
    .collect()
)

actual_exception_status = {
    row["event_id"]: row["is_temperature_exception"]
    for row in actual_exception_rows
}

for event_id, expected_status in expected_exception_status.items():
    actual_status = actual_exception_status.get(event_id)

    print(
        event_id,
        "expected =",
        expected_status,
        "actual =",
        actual_status
    )

    assert actual_status == expected_status, (
        f"{event_id}: expected {expected_status}, "
        f"found {actual_status}"
    )

total_silver_exceptions = (
    enriched_sensors
    .filter(
        F.col("is_temperature_exception") == True
    )
    .count()
)

assert total_silver_exceptions == 3

print()
print("5. TEMPERATURE WINDOW RECONCILIATION")
print("----------------------------------------------")

gold_accepted_readings = (
    temperature_windows
    .agg(
        F.sum("accepted_reading_count").alias("total")
    )
    .first()["total"]
)

gold_exception_count = (
    temperature_windows
    .agg(
        F.sum("exception_count").alias("total")
    )
    .first()["total"]
)

print("Enriched Silver readings:", enriched_sensors.count())
print("Gold accepted readings  :", gold_accepted_readings)
print("Silver exceptions       :", total_silver_exceptions)
print("Gold exceptions         :", gold_exception_count)

assert gold_accepted_readings == enriched_sensors.count(), (
    "Gold accepted-reading count does not reconcile "
    "with enriched Silver"
)

assert gold_exception_count == total_silver_exceptions, (
    "Gold exception count does not reconcile "
    "with enriched Silver"
)

invalid_exception_rates = (
    temperature_windows
    .filter(
        (F.col("exception_rate") < 0)
        | (F.col("exception_rate") > 100)
    )
    .count()
)

print("Invalid exception rates :", invalid_exception_rates)

assert invalid_exception_rates == 0, (
    f"Found {invalid_exception_rates} invalid exception rates"
)

invalid_window_counts = (
    temperature_windows
    .filter(
        (F.col("accepted_reading_count") <= 0)
        | (F.col("exception_count") < 0)
        | (
            F.col("exception_count")
            > F.col("accepted_reading_count")
        )
    )
    .count()
)

print("Invalid window counts   :", invalid_window_counts)

assert invalid_window_counts == 0, (
    f"Found {invalid_window_counts} invalid Gold window counts"
)

print("Temperature-window reconciliation: PASS")

print()
print("6. REFERENTIAL INTEGRITY")
print("----------------------------------------------")

sensor_without_shipment = (
    silver_sensors.alias("se")
    .join(
        silver_shipments.alias("sh"),
        F.col("se.shipment_id") == F.col("sh.shipment_id"),
        "left_anti"
    )
    .count()
)

shipment_without_product = (
    silver_shipments.alias("sh")
    .join(
        silver_products.alias("p"),
        F.col("sh.product_id") == F.col("p.product_id"),
        "left_anti"
    )
    .count()
)

delivery_without_shipment = (
    silver_deliveries.alias("d")
    .join(
        silver_shipments.alias("sh"),
        F.col("d.shipment_id") == F.col("sh.shipment_id"),
        "left_anti"
    )
    .count()
)

print("Sensors without shipment   :", sensor_without_shipment)
print("Shipments without product  :", shipment_without_product)
print("Deliveries without shipment:", delivery_without_shipment)

assert sensor_without_shipment == 0, (
    "Silver sensors without a matching Silver shipment were found"
)

assert shipment_without_product == 0, (
    "Silver shipments without a matching Silver product were found"
)

assert delivery_without_shipment == 0, (
    "Silver deliveries without a matching Silver shipment were found"
)

print("Referential integrity: PASS")

print()
print("7. SHIPMENT HEALTH INTEGRITY")
print("----------------------------------------------")

health_duplicate_shipments = (
    shipment_health
    .groupBy("shipment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate shipment-health records:",
    health_duplicate_shipments
)

assert health_duplicate_shipments == 0, (
    "Duplicate shipment IDs found in shipment_health"
)

assert shipment_health.count() == silver_shipments.count(), (
    "shipment_health must contain one row per Silver shipment"
)

invalid_active_missing_status = (
    shipment_health
    .filter(
        (F.col("active_shipment_status") == "ACTIVE")
        & (F.col("missing_sensor_status") == "NOT_APPLICABLE")
    )
    .count()
)

print(
    "Invalid active missing-sensor statuses:",
    invalid_active_missing_status
)

assert invalid_active_missing_status == 0, (
    "An active shipment has NOT_APPLICABLE "
    "as its missing-sensor status"
)

sh003_health = (
    shipment_health
    .filter(F.col("shipment_id") == "SH003")
    .first()
)

assert sh003_health is not None, (
    "SH003 was not found in shipment_health"
)

assert sh003_health["active_shipment_status"] == "ACTIVE", (
    "SH003 should be ACTIVE because no delivery exists"
)

assert sh003_health["actual_arrival"] is None, (
    "SH003 must not have a manufactured actual arrival"
)

print(
    "SH003 active status       :",
    sh003_health["active_shipment_status"]
)

print(
    "SH003 missing sensor status:",
    sh003_health["missing_sensor_status"]
)

print("Shipment-health integrity: PASS")

print()
print("8. DELIVERY PERFORMANCE")
print("----------------------------------------------")

expected_delays = {
    "SH001": 30,
    "SH002": 0,
    "SH004": 0,
    "SH005": 60
}

delivery_results = {
    row["shipment_id"]: row
    for row in delivery_performance.collect()
}

for shipment_id, expected_delay in expected_delays.items():
    assert shipment_id in delivery_results, (
        f"{shipment_id} is missing from delivery_performance"
    )

    actual_delay = delivery_results[shipment_id]["delay_minutes"]

    print(
        shipment_id,
        "expected =",
        expected_delay,
        "actual =",
        actual_delay
    )

    assert actual_delay == expected_delay, (
        f"{shipment_id}: expected delay {expected_delay}, "
        f"found {actual_delay}"
    )

assert "SH003" in delivery_results, (
    "SH003 is missing from delivery_performance"
)

sh003_delivery = delivery_results["SH003"]

assert sh003_delivery["actual_arrival"] is None, (
    "SH003 actual arrival must remain NULL"
)

assert sh003_delivery["delay_minutes"] is None, (
    "SH003 delay must remain NULL because it is undelivered"
)

assert sh003_delivery["delivery_status"] == "UNDELIVERED", (
    "SH003 should have delivery_status UNDELIVERED"
)

assert (
    sh003_delivery["open_overdue_status"] == "OPEN_OVERDUE"
), (
    "SH003 should be OPEN_OVERDUE for the selected evaluation time"
)

negative_delivery_delays = (
    delivery_performance
    .filter(F.col("delay_minutes") < 0)
    .count()
)

print("Negative delivery delays:", negative_delivery_delays)

assert negative_delivery_delays == 0, (
    f"Found {negative_delivery_delays} negative delivery delays"
)

print("Delivery-performance integrity: PASS")

print()
print("9. GOLD RUN METADATA")
print("----------------------------------------------")

gold_tables = {
    "temperature_windows": temperature_windows,
    "shipment_health": shipment_health,
    "delivery_performance": delivery_performance
}

for table_name, dataframe in gold_tables.items():
    required_metadata_columns = {
        "run_id",
        "evaluated_at"
    }

    missing_columns = (
        required_metadata_columns - set(dataframe.columns)
    )

    assert not missing_columns, (
        f"{table_name} is missing metadata columns: "
        f"{sorted(missing_columns)}"
    )

    missing_run_id = (
        dataframe
        .filter(
            F.col("run_id").isNull()
            | (F.trim(F.col("run_id")) == "")
        )
        .count()
    )

    missing_evaluated_at = (
        dataframe
        .filter(
            F.col("evaluated_at").isNull()
        )
        .count()
    )

    print(
        table_name,
        "missing run IDs =",
        missing_run_id,
        "missing evaluation times =",
        missing_evaluated_at
    )

    assert missing_run_id == 0, (
        f"{table_name} contains "
        f"{missing_run_id} missing run IDs"
    )

    assert missing_evaluated_at == 0, (
        f"{table_name} contains "
        f"{missing_evaluated_at} missing evaluation timestamps"
    )

print("Gold run metadata: PASS")

print()
print("==============================================")
print("FINAL GOLD ACCEPTANCE REPORT")
print("==============================================")

print(
    "Bronze sensor rows        :",
    counts["bronze_sensors"]
)

print(
    "Silver sensor rows        :",
    counts["silver_sensors"]
)

print(
    "Quarantine sensor rows    :",
    counts["quarantine_sensors"]
)

print(
    "Removed duplicate copies  :",
    duplicate_copies_removed
)

print(
    "Enriched sensor rows      :",
    counts["enriched_sensors"]
)

print(
    "Temperature exceptions    :",
    total_silver_exceptions
)

print(
    "Gold temperature windows  :",
    temperature_windows.count()
)

print(
    "Shipment health rows      :",
    shipment_health.count()
)

print(
    "Delivery performance rows :",
    delivery_performance.count()
)

print()
print("GOLD TABLE DETAILS")
print("----------------------------------------------")

for table_name, dataframe in gold_tables.items():
    record_count = dataframe.count()

    run_ids = [
        row["run_id"]
        for row in (
            dataframe
            .select("run_id")
            .distinct()
            .orderBy("run_id")
            .collect()
        )
    ]

    earliest_evaluation = (
        dataframe
        .agg(
            F.min("evaluated_at").alias(
                "earliest_evaluation"
            )
        )
        .first()["earliest_evaluation"]
    )

    latest_evaluation = (
        dataframe
        .agg(
            F.max("evaluated_at").alias(
                "latest_evaluation"
            )
        )
        .first()["latest_evaluation"]
    )

    print()
    print("Table name          :", table_name)
    print("Record count        :", record_count)
    print("Run IDs             :", run_ids)
    print("Earliest evaluation :", earliest_evaluation)
    print("Latest evaluation   :", latest_evaluation)

print()
print("==============================================")
print("FINAL GOLD ACCEPTANCE STATUS: PASS")
print("==============================================")